# 태그 조합 분석

**분석 대상**
- `steam_indie_games.csv` — 인디게임 전체 데이터

**분석 질문**
- 어떤 태그가 통계적으로 높은 긍정률과 연관되는가?
- 장르 내에서 차별화에 도움이 되는 태그 조합은 무엇인가?
- 출시 전 태그 설정 시 참고할 수 있는 신뢰도 높은 고긍정률 태그 상위 20개는?

**분석 방법**
- 윌슨 신뢰구간 하한값(Wilson Score Lower Bound)으로 태그별 복합 지표 산출
- 리뷰 수가 적은 태그는 자동으로 패널티 적용 → 통계적으로 검증된 상위 20개 선정

In [1]:
import ast
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

print('라이브러리 로드 완료')

라이브러리 로드 완료


In [2]:
DATA_PATH = Path('../../../data/preprocessed/steam_indie_games.csv')
df_games = pd.read_csv(DATA_PATH)
print(f'데이터 로드 완료: {df_games.shape[0]:,}개 게임')

데이터 로드 완료: 8,730개 게임


In [3]:
def parse_tags(value: str) -> dict[str, int]:
    if pd.isna(value):
        return {}
    try:
        return json.loads(value)
    except (json.JSONDecodeError, TypeError):
        return {}


def top_tags(tag_dict: dict, n: int = 5) -> list[str]:
    """투표수 기준 상위 n개 태그를 반환합니다."""
    return sorted(tag_dict, key=tag_dict.get, reverse=True)[:n]


def wilson_lower(positive_sum: float, total_sum: float, z: float = 1.96) -> float:
    """윌슨 신뢰구간 하한값 (95% 신뢰수준).

    태그에 속한 게임들의 총 긍정 리뷰 수와 총 리뷰 수를 기반으로
    '이 태그의 실제 긍정률이 최소 얼마 이상'인지를 보수적으로 추정한다.
    """
    if total_sum == 0:
        return 0.0
    p = positive_sum / total_sum
    n = total_sum
    return (
        (p + z**2 / (2 * n) - z * ((p * (1 - p) + z**2 / (4 * n)) / n) ** 0.5)
        / (1 + z**2 / n)
    ) * 100


df_clean = df_games.copy()
for col in ['positive', 'negative', 'total_reviews']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')

df_clean = df_clean.dropna(subset=['positive', 'negative', 'total_reviews'])
df_clean = df_clean[df_clean['total_reviews'] > 0].copy()
df_clean['positive_rate'] = df_clean['positive'] / df_clean['total_reviews'] * 100
df_clean['tag_dict'] = df_clean['tags'].apply(parse_tags)
df_clean['top_tags'] = df_clean['tag_dict'].apply(lambda d: top_tags(d, n=5))
df_clean = df_clean[df_clean['top_tags'].map(len) > 0].copy()

print(f'분석 가능 게임 수: {len(df_clean):,}개')

분석 가능 게임 수: 6,017개


## 전체 게임 대비 태그 비중 Top 10

In [13]:
TOTAL_GAMES = df_clean['appid'].nunique()

# 데이터에 존재하는 모든 장르명 수집 (태그 제외 기준)
all_genres = set(
    df_clean['genres']
    .dropna()
    .apply(parse_genres)
    .explode()
    .str.strip()
    .dropna()
    .unique()
)

tag_freq = (
    df_clean.explode('top_tags')
    .rename(columns={'top_tags': 'tag'})
    .assign(tag=lambda d: d['tag'].astype(str).str.strip())
    .query("tag != ''")
    .loc[lambda d: ~d['tag'].isin(all_genres)]
    .groupby('tag')['appid']
    .nunique()
    .reset_index(name='game_count')
    .assign(proportion=lambda d: d['game_count'] / TOTAL_GAMES * 100)
    .sort_values('proportion', ascending=False)
    .head(10)
    .reset_index(drop=True)
)

print(f'전체 분석 게임 수: {TOTAL_GAMES:,}개')
print(f'제외된 장르 태그: {sorted(all_genres)}\n')
display(tag_freq[['tag', 'game_count', 'proportion']].rename(
    columns={'tag': '태그', 'game_count': '게임 수', 'proportion': '비중 (%)'}
).round(2))

fig = px.bar(
    tag_freq.sort_values('proportion'),
    x='proportion',
    y='tag',
    orientation='h',
    color='proportion',
    color_continuous_scale='Blues',
    text='proportion',
    title='전체 게임 대비 태그 비중 Top 10',
    labels={'proportion': '비중 (%)', 'tag': '태그'},
    hover_data={'game_count': True},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=450)
fig.show()

전체 분석 게임 수: 6,017개
제외된 장르 태그: ['Action', 'Adventure', 'Casual', 'Indie', 'RPG', 'Racing', 'Simulation', 'Sports', 'Strategy']



,태그,게임 수,비중 (%)
0,Puzzle,876,14.56
1,Exploration,640,10.64
2,Horror,482,8.01
3,2D,462,7.68
4,Pixel Graphics,458,7.61
5,Action Roguelike,429,7.13
6,Singleplayer,426,7.08
7,Rogue-lite,395,6.56
8,Arcade,392,6.51
9,Action-Adventure,386,6.42


## 태그별 윌슨 스코어 상위 20개

In [4]:
MIN_GAMES_PER_TAG = 30
TOP_N_TAGS = 20

df_tag = df_clean.explode('top_tags').rename(columns={'top_tags': 'tag'})
df_tag['tag'] = df_tag['tag'].astype(str).str.strip()
df_tag = df_tag[df_tag['tag'] != ''].copy()

tag_stats = (
    df_tag
    .groupby('tag')
    .agg(
        game_count=('appid', 'nunique'),
        positive_sum=('positive', 'sum'),
        total_reviews_sum=('total_reviews', 'sum'),
        avg_positive_rate=('positive_rate', 'mean'),
        median_positive_rate=('positive_rate', 'median'),
        median_reviews=('total_reviews', 'median'),
    )
    .reset_index()
)

tag_stats['wilson_score'] = tag_stats.apply(
    lambda r: wilson_lower(r['positive_sum'], r['total_reviews_sum']), axis=1
)

tag_stats_filtered = (
    tag_stats[tag_stats['game_count'] >= MIN_GAMES_PER_TAG]
    .sort_values('wilson_score', ascending=False)
    .reset_index(drop=True)
)

print(f'게임 수 {MIN_GAMES_PER_TAG}개 이상 태그: {len(tag_stats_filtered)}개')
display(
    tag_stats_filtered
    .head(10)[['tag', 'game_count', 'total_reviews_sum', 'avg_positive_rate', 'wilson_score']]
    .round(2)
)

게임 수 30개 이상 태그: 186개


,tag,game_count,total_reviews_sum,avg_positive_rate,wilson_score
0,Sokoban,45,2912,95.37,95.95
1,Multiple Endings,87,117579,87.55,95.75
2,Automation,53,271080,82.43,95.72
3,Wholesome,52,23517,91.75,95.20
4,Cyberpunk,49,59815,86.17,95.11
5,Minimalist,63,30155,92.08,94.90
6,Visual Novel,308,182853,89.41,94.41
7,Roguelike Deckbuilder,96,243797,84.40,94.35
8,Crafting,74,271680,74.77,94.29
9,Choices Matter,106,106951,85.26,94.25


윌슨 스코어는 긍정률과 리뷰 수를 함께 반영한 보수적 신뢰도 지표다. 리뷰가 아무리 많아도 긍정률이 낮으면 스코어가 낮고, 긍정률이 높아도 리뷰가 적으면 패널티가 부여된다. 상위 태그는 "이 태그를 달면 95% 확률로 최소 이 수준의 긍정률을 기대할 수 있다"는 의미다.

In [9]:
plot_df = tag_stats_filtered.head(TOP_N_TAGS).sort_values('wilson_score')

fig = px.bar(
    plot_df,
    x='wilson_score',
    y='tag',
    orientation='h',
    color='wilson_score',
    color_continuous_scale='Blues',
    text='wilson_score',
    title=f'태그별 상위 {TOP_N_TAGS}개 (게임 수 {MIN_GAMES_PER_TAG}개 이상)',
    labels={'wilson_score': '윌슨 스코어 (하한 긍정률, %)', 'tag': '태그'},
    range_x=[0, 100],
    hover_data={'avg_positive_rate': ':.1f', 'game_count': True, 'total_reviews_sum': True},
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(coloraxis_showscale=False, height=600)
fig.show()

상위 태그들은 단순히 긍정률이 높은 것이 아니라, 충분한 리뷰 볼륨으로 신뢰도까지 검증된 태그다. 인디 개발사는 이 목록에서 자신의 게임 장르·분위기와 맞는 태그를 우선 고려할 수 있으며, 특히 상위권 태그는 유저 만족도가 안정적으로 높은 카테고리임을 의미한다.

## 장르별 상위 태그 긍정률 히트맵

In [6]:
def parse_genres(value: str) -> list[str]:
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(item).strip() for item in parsed if str(item).strip()]
    except (ValueError, SyntaxError):
        pass
    return [item.strip() for item in str(value).split(',') if item.strip()]


df_clean['genre_list'] = df_clean['genres'].apply(parse_genres)
df_genre_tag = (
    df_clean
    .explode('genre_list')
    .rename(columns={'genre_list': 'genre'})
)
df_genre_tag['genre'] = df_genre_tag['genre'].astype(str).str.strip()
df_genre_tag = df_genre_tag[
    (df_genre_tag['genre'] != '') & (df_genre_tag['genre'] != 'Indie')
].copy()
df_genre_tag = df_genre_tag.explode('top_tags').rename(columns={'top_tags': 'tag'})
df_genre_tag['tag'] = df_genre_tag['tag'].astype(str).str.strip()

# 윌슨 스코어 기준 상위 20개 태그
top_tags_list = tag_stats_filtered.head(TOP_N_TAGS)['tag'].tolist()
df_genre_tag = df_genre_tag[df_genre_tag['tag'].isin(top_tags_list)].copy()

MIN_CELL = 5
cell_data = (
    df_genre_tag
    .groupby(['genre', 'tag'])
    .agg(avg_positive_rate=('positive_rate', 'mean'), count=('appid', 'nunique'))
    .reset_index()
)
cell_data.loc[cell_data['count'] < MIN_CELL, 'avg_positive_rate'] = float('nan')
pivot = cell_data.pivot(index='genre', columns='tag', values='avg_positive_rate').round(1)

fig = px.imshow(
    pivot,
    text_auto='.1f',
    color_continuous_scale='RdYlGn',
    zmin=70,
    zmax=100,
    title=f'장르 × 태그별 평균 긍정률 (게임 수 {MIN_CELL}개 미만 셀 제외)',
    labels={'x': '태그', 'y': '장르', 'color': '평균 긍정률 (%)'},
    aspect='auto',
)
fig.update_layout(height=400)
fig.show()

장르별로 윌슨 스코어 상위 태그의 긍정률 분포를 확인할 수 있다. 셀이 비어 있는 경우는 해당 장르에서 그 태그를 가진 게임이 5개 미만이라 통계적으로 신뢰하기 어렵다는 의미다. 특정 장르에서 유독 높은 긍정률을 보이는 태그 조합은 해당 장르 내 차별화 전략으로 활용할 수 있다.

## 태그별 총 리뷰 수 vs 윌슨 스코어 산점도

In [7]:
plot_scatter = tag_stats_filtered.head(50)

fig = px.scatter(
    plot_scatter,
    x='total_reviews_sum',
    y='wilson_score',
    size='game_count',
    color='wilson_score',
    color_continuous_scale='RdYlGn',
    text='tag',
    size_max=40,
    title='태그별 총 리뷰 수 vs 윌슨 스코어 (원 크기: 게임 수)',
    labels={
        'total_reviews_sum': '총 리뷰 수 (투표 규모)',
        'wilson_score': '윌슨 스코어 (하한 긍정률, %)',
        'game_count': '게임 수',
    },
    hover_data={'avg_positive_rate': ':.1f', 'game_count': True},
    log_x=True,
)
fig.update_traces(
    textposition='top center',
    textfont=dict(size=9),
)
fig.update_layout(height=550, coloraxis_showscale=False)
fig.show()

x축(총 리뷰 수)은 태그의 시장 규모를, y축(윌슨 스코어)은 만족도 신뢰도를 나타낸다. 오른쪽 상단에 위치한 태그일수록 리뷰 볼륨도 크고 긍정률도 높아 가장 매력적인 포지셔닝이다. 반대로 리뷰는 많지만 윌슨 스코어가 낮은 태그는 시장은 크지만 경쟁이 치열하거나 유저 기대치를 충족하기 어려운 카테고리다.

## 요약 테이블

In [8]:
print(f'태그별 윌슨 스코어 요약 (상위 {TOP_N_TAGS}개)')
display(
    tag_stats_filtered
    .head(TOP_N_TAGS)[['tag', 'game_count', 'total_reviews_sum', 'avg_positive_rate', 'median_positive_rate', 'wilson_score']]
    .round(2)
    .reset_index(drop=True)
)

태그별 윌슨 스코어 요약 (상위 20개)


,tag,game_count,total_reviews_sum,avg_positive_rate,median_positive_rate,wilson_score
0,Sokoban,45,2912,95.37,96.88,95.95
1,Multiple Endings,87,117579,87.55,90.91,95.75
2,Automation,53,271080,82.43,83.89,95.72
3,Wholesome,52,23517,91.75,96.19,95.20
4,Cyberpunk,49,59815,86.17,91.43,95.11
5,Minimalist,63,30155,92.08,95.65,94.90
6,Visual Novel,308,182853,89.41,92.19,94.41
7,Roguelike Deckbuilder,96,243797,84.40,87.13,94.35
8,Crafting,74,271680,74.77,78.97,94.29
9,Choices Matter,106,106951,85.26,89.70,94.25


## 출시 전 태그 전략 제안

윌슨 스코어 기준 상위 20개 태그는 통계적으로 검증된 고만족 카테고리다. 인디 개발사가 태그를 설정할 때 참고할 수 있는 원칙:

- **장르·분위기가 맞다면 상위권 태그를 우선 선택**: 윌슨 스코어가 높을수록 유저 기대치 충족 가능성이 높은 카테고리로, 태그를 통한 첫인상이 긍정적으로 형성될 가능성이 크다.
- **총 리뷰 수와 윌슨 스코어를 함께 고려**: 리뷰 볼륨이 크면서 윌슨 스코어도 높은 태그는 시장 규모와 만족도를 동시에 잡을 수 있는 카테고리다.
- **장르 히트맵 활용**: 자신의 게임이 속한 장르에서 특히 높은 긍정률을 보이는 태그 조합을 우선 검토한다. 같은 태그라도 장르에 따라 유저 반응이 다를 수 있다.